### Practice: Parameter Efficient Fine-Tuning
In this notebook, you're gonna fine-tune large language models within limited GPU memory.

In [2]:
import gc

import torch
import torch.nn as nn
import torch.nn.functional as F

import transformers
from tqdm.auto import tqdm, trange

assert torch.cuda.is_available(), "you need cuda for this part"
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

In [3]:
model_name = 'unsloth/Qwen3-8B-Base-bnb-4bit'
tokenizer = transformers.AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token_id = tokenizer.eos_token_id

# the main model weights are loaded in 4-bit precision - but we can still tune LoRAs later
model = transformers.AutoModelForCausalLM.from_pretrained(
    model_name, device_map='auto', low_cpu_mem_usage=True, dtype=torch.float32,
    quantization_config=transformers.BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4"))

for param in model.parameters():
    param.requires_grad=False
model.gradient_checkpointing_enable()  # only store a small subset of activations, re-compute the rest.
model.enable_input_require_grads()     # override an implementation quirk in gradient checkpoints that disables backprop unless inputs require grad
# more on gradient checkpointing: https://pytorch.org/docs/stable/checkpoint.html https://arxiv.org/abs/1604.06174
model.config.use_cache = False
torch.manual_seed(1337)

/usr/local/lib/python3.13/dist-packages/transformers/quantizers/auto.py:279: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.
  warnings.warn(warning_msg)


model.safetensors: reconstructing file:   0%|          |  0.00B / 6.07GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/166 [00:00<?, ?B/s]

### Prompt tuning: the story of a fox (1 point)

![img](https://i.imgur.com/Ux3qQAu.png) (source: theodd1souts.fandom.com)

In [4]:
prompt = 'A quick brown fox'
batch = tokenizer(prompt, return_tensors='pt', return_token_type_ids=False).to(device)

model.eval()
with torch.no_grad():
    for i in range(10):
        next_token = model(**batch).logits[0, -1].argmax(-1).reshape(1, 1)
        batch['input_ids'] = torch.cat([batch['input_ids'], next_token], dim=-1)
        batch['attention_mask'] = torch.cat([batch['attention_mask'], torch.ones_like(next_token)], dim=-1)

print("\nOutput:", tokenizer.decode(batch['input_ids'][0].cpu().numpy().tolist()))


Output: A quick brown fox jumps over the lazy dog. The quick brown fox


What a blatant lie! This particular fox assures you that it didn't in fact jump over the lazy dog. No, sir! The fox was just minding its own business. __Your task is to train the model to say truth: no dog was jumped over today.__

In [5]:
the_truth = "A quick brown fox did not jump over the lazy dog. Besides, that dog deserved it anyway!"
batch = tokenizer(the_truth, return_tensors='pt', return_token_type_ids=False).to(device)
with torch.no_grad():
    outputs = model(**batch)

next_word_logits = outputs.logits[:, :-1]
true_next_tokens = batch['input_ids'][:, 1:]
loss = F.cross_entropy(next_word_logits.flatten(0, 1), true_next_tokens.flatten(0, 1))

print("Loss:", loss)

Loss: tensor(3.5625, device='cuda:0', dtype=torch.bfloat16)


Except, we can't train the entire model - that would be 28GB gradients in float32. Instead, let's run [prompt tuning](https://arxiv.org/abs/2104.08691).

![img](https://i.imgur.com/VwNNKnb.png)


In [6]:
class WordEmbeddingsWithLearnedPrompts(nn.Module):
    """
    To perform prompt tuning, you will need to replace model's original word embeddings with a layer - THIS layer
     - that inserts trainable prompts instead of the first N token embeddings. """

    def __init__(self, word_embeddings: nn.Embedding, num_prompts: int):
        super().__init__()
        self.original_word_embeddings = word_embeddings
        self.num_prompts = num_prompts
        self.learnable_prompts = nn.Parameter(
            torch.randn(1, num_prompts, word_embeddings.embedding_dim), requires_grad=True)

    def forward(self, input_ids: torch.LongTensor):
        # input_ids shape: [batch_size, seq length]
        assert input_ids.dtype == torch.int64
        assert input_ids.shape[1] > self.num_prompts
        assert torch.all(input_ids[:, :self.num_prompts] == tokenizer.pad_token_id).item(), "don't forget to prepend several BOS tokens to input_ids"

        # Your task: embed input_ids, but replace the first :num_prompts: tokens with self.learnable_prompts
        # This is because we will prepend :num_prompts: padding tokens at the beginning

        # After you are done, you must produce a word embedding vector for each token in input_ids,
        # except that the first :num_prompts: vectors should equal learnable_prompts;
        # any additional vectors after first :num_prompts: ones should be embedded as usual
        # Note: since you're dealing with trainable params, please torch.cat instead of item assignment

        embeddings = self.original_word_embeddings(input_ids[:, self.num_prompts:])
        prompts = self.learnable_prompts.expand(input_ids.shape[0], -1, -1)
        return torch.cat([prompts.to(embeddings.dtype), embeddings], dim=1)

In [7]:
num_prompts = 16
test_emb_layer = WordEmbeddingsWithLearnedPrompts(model.model.embed_tokens, num_prompts=num_prompts).to(device)
test_input_ids = tokenizer("a cat say on a may", return_tensors='pt')['input_ids'].to(device)

space_for_prompts = torch.full([len(test_input_ids), num_prompts], fill_value=tokenizer.pad_token_id,
                               dtype=torch.int64, device=device)
test_inputs_with_prompts = torch.cat([space_for_prompts, test_input_ids], dim=1)

with torch.no_grad():
    test_prompt_embeddings = test_emb_layer(test_inputs_with_prompts)
    expected_prompts = test_emb_layer.learnable_prompts.to(test_prompt_embeddings)
    expected_embeddings = model.model.embed_tokens(test_input_ids).to(test_prompt_embeddings)

assert test_prompt_embeddings.shape[:2] == test_inputs_with_prompts.shape
assert test_prompt_embeddings.shape[-1] == model.config.hidden_size
assert torch.allclose(test_prompt_embeddings[:, :num_prompts], expected_prompts)
assert torch.allclose(test_prompt_embeddings[:, num_prompts:], expected_embeddings)
print("Looks legit!")

Looks legit!


__Now that it works,__ let's inject learnable prompts into the main model and teach it about foxes.

In [8]:
assert isinstance(model.model.embed_tokens, nn.Embedding), "you have already replaced the embedding layer. If the replacement is broken, please reload the model"

model.model.embed_tokens = WordEmbeddingsWithLearnedPrompts(model.model.embed_tokens, num_prompts=num_prompts).to(device)

opt = torch.optim.Adam([model.model.embed_tokens.learnable_prompts], lr=0.01)

In [9]:
the_truth = "A quick brown fox did not jump over the lazy dog. Besides, that dog deserved it anyway!"
batch = tokenizer(the_truth, return_tensors='pt', return_token_type_ids=False).to(device)
space_for_prompts = torch.full([len(test_input_ids), num_prompts], fill_value=tokenizer.pad_token_id,
                               dtype=torch.int64, device=device)
batch['input_ids'] = torch.cat([space_for_prompts, batch['input_ids']], dim=1)
batch['attention_mask'] = torch.cat([torch.ones_like(space_for_prompts), batch['attention_mask']], dim=1)

outputs = model(**batch)
next_word_logits = outputs.logits[:, num_prompts : -1, :]
true_next_tokens = batch['input_ids'][:, num_prompts + 1:]
loss = F.cross_entropy(next_word_logits.flatten(0, 1), true_next_tokens.flatten(0, 1))
print("Loss:", loss)


# trained only the virtual token embeddings
model.train()
prompt_loss_history = []
for step in trange(500, desc='Manual prompt tuning'):
    opt.zero_grad()
    outputs = model(**batch)
    next_word_logits = outputs.logits[:, num_prompts:-1, :]
    true_next_tokens = batch['input_ids'][:, num_prompts + 1:]
    loss = F.cross_entropy(next_word_logits.flatten(0, 1), true_next_tokens.flatten(0, 1))
    prompt_loss_history.append(loss.item())
    if step % 25 == 0:
        print(f'step {step}: loss = {loss.item():.4f}')
    if loss.item() <= 0.1:
        break
    loss.backward()
    opt.step()

model.eval()
with torch.no_grad():
    outputs = model(**batch)
    loss = F.cross_entropy(
        outputs.logits[:, num_prompts:-1, :].flatten(0, 1),
        batch['input_ids'][:, num_prompts + 1:].flatten(0, 1)
    )
print(f'final manual prompt loss = {loss.item():.4f}')


assert loss.item() <= 0.1
print("Good job!")

Loss: tensor(3.8125, device='cuda:0', dtype=torch.bfloat16,
       grad_fn=<NllLossBackward0>)


Manual prompt tuning:   0%|          | 0/500 [00:00<?, ?it/s]

step 0: loss = 3.8125
step 25: loss = 0.8633
final manual prompt loss = 0.0466
Good job!


In [10]:
prompt = 'A quick brown fox'
batch = tokenizer(prompt, return_tensors='pt', return_token_type_ids=False).to(device)
batch['input_ids'] = torch.cat([space_for_prompts, batch['input_ids']], dim=1)
batch['attention_mask'] = torch.cat([torch.ones_like(space_for_prompts), batch['attention_mask']], dim=1)


model.eval()
with torch.no_grad():
    for i in range(15):
        next_token = model(**batch).logits[0, -1].argmax(-1).reshape(1, 1)
        batch['input_ids'] = torch.cat([batch['input_ids'], next_token], dim=-1)
        batch['attention_mask'] = torch.cat([batch['attention_mask'], torch.ones_like(next_token)], dim=-1)

print("\nOutput:", tokenizer.decode(batch['input_ids'][0, num_prompts:].cpu().numpy().tolist()))

# if you did everything right, the model will deny that the fox jumped over the lazy dog


Output: A quick brown fox did not jump over the lazy dog. Besides, that dog deserved it anyway


### Using HuggingFace PEFT

[`peft`](https://huggingface.co/docs/peft/index) is a transformer's sister library that allows you to apply various __p__arameter __e__fficient __f__ine-__t__uning methods to pre-trained transformers. The library imlements both prompt tuning, prefix tuning, as well as several adapter-based techniques under a common interface:



In [11]:
import peft
# restored the unchanged backbone before the library-based experiment
model.model.embed_tokens = model.model.embed_tokens.original_word_embeddings
del opt, test_emb_layer, outputs, next_word_logits, loss
gc.collect()
torch.cuda.empty_cache()
assert isinstance(model.model.embed_tokens, nn.Embedding), "please reload the model"

peft_config = peft.PromptTuningConfig(task_type=peft.TaskType.CAUSAL_LM, num_virtual_tokens=16)
model = peft.get_peft_model(model, peft_config)  # note: for most peft methods, this line also modifies model in-place
print("Trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))
print("Total parameters (excluding quantization):", sum(p.numel() for p in model.parameters()))

Trainable parameters: 65536
Total parameters (excluding quantization): 4717917184


In [12]:
# Your task: optimize the PEFT-wrapped model to achieve next token prediction loss < 0.1, but this time using PEFT
# Please note: you no longer need to prepend PAD tokens, but you still need to skip :num_virtual_tokens: first logits.
# Finally, generate the sentence to make sure that the model learned the truth.

peft_opt = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=0.01)
batch = tokenizer(the_truth, return_tensors='pt', return_token_type_ids=False).to(device)
virtual_tokens = peft_config.num_virtual_tokens
peft_loss_history = []
model.train()
model.config.use_cache = False
for step in trange(500, desc='PEFT prompt tuning'):
    peft_opt.zero_grad()
    outputs = model(**batch)
    logits = outputs.logits[:, virtual_tokens:-1, :]
    loss = F.cross_entropy(logits.flatten(0, 1), batch['input_ids'][:, 1:].flatten(0, 1))
    peft_loss_history.append(loss.item())
    if step % 25 == 0:
        print(f'step {step}: loss = {loss.item():.4f}')
    if loss.item() < 0.1:
        break
    loss.backward()
    peft_opt.step()

model.eval()
with torch.no_grad():
    outputs = model(**batch)
    loss = F.cross_entropy(
        outputs.logits[:, virtual_tokens:-1, :].flatten(0, 1),
        batch['input_ids'][:, 1:].flatten(0, 1)
    )
print(f'final PEFT prompt loss = {loss.item():.4f}')
assert loss.item() < 0.1

PEFT prompt tuning:   0%|          | 0/500 [00:00<?, ?it/s]

step 0: loss = 3.9062
step 25: loss = 2.3906
final PEFT prompt loss = 0.0762


In [13]:
# Feel free to structure your code as you see fit - as long as it's legible :)
prompt = 'A quick brown fox'
batch = tokenizer(prompt, return_tensors='pt', return_token_type_ids=False).to(device)
model.eval()
# recomputed virtual prompts without using a generation cache
with torch.no_grad():
    for i in range(15):
        next_token = model(**batch, use_cache=False).logits[:, -1].argmax(-1, keepdim=True)
        batch['input_ids'] = torch.cat([batch['input_ids'], next_token], dim=1)
        batch['attention_mask'] = torch.cat([batch['attention_mask'], torch.ones_like(next_token)], dim=1)
print(tokenizer.decode(batch['input_ids'][0], skip_special_tokens=True))

A quick brown fox did not jump over the lazy dog. Besides, that dog deserved it anyway


### Parameter-efficient finetuning with LoRA (1 point)

When training on more serious tasks, you can use low-rank adapters based on the [LoRA paper](https://arxiv.org/pdf/2106.09685.pdf).

The core idea is to add low-rank adapters __in parallel with existing linear layers,__ like this:
<center><img src="https://i.imgur.com/6bQLNiG.png" width=240px></center>

In the original LoRA paper, the adapters were only added to attention projection matrices. However, [subsequent works](https://arxiv.org/abs/2305.14314) show that it is useful to adapt FFNs as well. But before we do any training, we need to implement the basic LoRA layer.

In [14]:
import peft  # v-- re-load the notebook to remove any previous PEFT tuners, run cell 1 and this
# released old models and optimizers before the independent LoRA run
for variable in ('code_trainer', 'trainer', 'model', 'peft_opt', 'opt', 'outputs', 'logits', 'loss'):
    globals().pop(variable, None)
gc.collect()
torch.cuda.empty_cache()
model_name = 'unsloth/Qwen3-8B-Base-bnb-4bit'
tokenizer = transformers.AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token_id = tokenizer.eos_token_id

# avoided FP16 for the retry and matched the quantized compute dtype
use_bf16 = torch.cuda.is_bf16_supported()
compute_dtype = torch.bfloat16 if use_bf16 else torch.float32
config = transformers.AutoConfig.from_pretrained(model_name)
config.quantization_config['bnb_4bit_compute_dtype'] = 'bfloat16' if use_bf16 else 'float32'
print('GPU:', torch.cuda.get_device_name(), '| retry compute dtype:', compute_dtype)
model = transformers.AutoModelForCausalLM.from_pretrained(
    model_name, config=config, device_map='auto',
    low_cpu_mem_usage=True, dtype=torch.float32,
)
model = peft.prepare_model_for_kbit_training(
    model, use_gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False}
)
model.config.use_cache = False


GPU: Tesla T4 | retry compute dtype: torch.bfloat16


Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

In [15]:
class LoRALayer(nn.Module):
    """Wraps a linear layer with LoRA-like adapter. Wraps an existing OPT linear layer"""
    def __init__(self, module: nn.Linear, rank: int):
        super().__init__()
        self.module = module  # pre-trained (frozen) linear layer
        self.adapter_A = nn.Parameter(torch.empty(module.in_features, rank, device=module.weight.device))
        nn.init.kaiming_uniform_(self.adapter_A, a=5 ** 0.5)
        self.adapter_B = nn.Parameter(torch.zeros(rank, module.out_features, device=module.weight.device))

    def forward(self, input):
        # Apply self.module and LoRA adapter, return the sum (self.module outputs + adapter outputs)
        base_output = self.module(input)
        adapter_output = (input.to(self.adapter_A.dtype) @ self.adapter_A) @ self.adapter_B
        return base_output + adapter_output.to(base_output.dtype)

In [16]:
# test your implementation
test_linear = nn.Linear(128, 128)
test_linear.weight.data[...] = torch.eye(128)
test_adapter = LoRALayer(test_linear, rank=8)

assert torch.allclose(test_adapter(torch.ones(1, 1, 128)), test_linear.bias + 1), "please check your forward pass"

test_adapter.adapter_A.data[...] = torch.linspace(0.1, -0.5, 128 * 8).view(128, 8)
test_adapter.adapter_B.data[...] = torch.linspace(0.5, -0.1, 128 * 8).view(8, 128)
test_linear.bias.data[...] = torch.linspace(1., -1., 128)

dummy_loss = F.mse_loss(test_adapter(torch.ones(1, 128) / 128).squeeze(), torch.linspace(-1, 1, 128))
assert torch.allclose(dummy_loss, torch.tensor(1.3711389), rtol=0, atol=1e-4)
dummy_loss.backward()
assert all(w.grad is not None for w in [test_adapter.adapter_A, test_adapter.adapter_B]), "some adapter weights have no grad"
assert torch.allclose(test_adapter.adapter_A.grad.sum(), torch.tensor(-0.60158), rtol=0, atol=1e-4), "bad grad w.r.t. A"
assert torch.allclose(test_adapter.adapter_B.grad.sum(), torch.tensor(0.9931), rtol=0, atol=1e-4), "bad grad w.r.t. B"
# note: bad grad means that your code is different from LoRA paper OR that your code is not autograd-friendly (e.g. no_grad)
del dummy_loss, test_linear, test_adapter
print("All tests passed!")

All tests passed!


### Apply LoRA to the model

The code below applies LoRA adapters on top of Q/K/V linear layers in attention blocks. You may also choose to modify other layers:
* self_attn.o_proj - attention output projection
* mlp.up_proj, mlp.gate_proj, mlp.down_proj - transformer feedforward layers
* lm_head - output LM head

__Note:__ please scroll down for the homework task

In [17]:
lora_rank = 8

for name, module in list(model.model.layers.named_modules()):
    if 'DecoderLayer' in repr(type(module)):
        module.self_attn.q_proj = LoRALayer(module.self_attn.q_proj, rank=lora_rank).to(device)
        module.self_attn.k_proj = LoRALayer(module.self_attn.k_proj, rank=lora_rank).to(device)
        module.self_attn.v_proj = LoRALayer(module.self_attn.v_proj, rank=lora_rank).to(device)
        # added the required extra adapter to the MLP
        module.mlp.down_proj = LoRALayer(module.mlp.down_proj, rank=lora_rank).to(device)

assert sum(isinstance(module, LoRALayer) for module in model.modules()) > 0, "Did not add any LoRA layers!"
print('Trainable parameters:', sum(p.numel() for p in model.parameters() if p.requires_grad))
assert all(
    'adapter_A' in name or 'adapter_B' in name
    for name, param in model.named_parameters() if param.requires_grad
)

Trainable parameters: 10027008


In [18]:
batch = tokenizer("This model wants to share its greatest secret:", return_tensors='pt', return_token_type_ids=False).to(device)
# test a single training step, make sure we get meaningful gradients
with torch.autocast(device_type='cuda', enabled=False):
    out = model.forward(**batch)
    (out.logits.norm() / 100).backward()

for i, module in enumerate(model.modules()):
    if isinstance(module, LoRALayer):
        assert module.adapter_B.grad is not None
        assert module.adapter_B.grad.norm().item() > 0

model.zero_grad(set_to_none=True)
print("Grad check successful, well done!")
del out

Grad check successful, well done!


### (example) How to train your model

The example below shows how to train the LoRA adapters on a dummy dataset. You will need to run a _similar_ training task later.

__Note:__ please scroll down for the homework task

In [19]:
# checking if the model can learn. Change max_steps for proper training
import datasets
data = datasets.load_dataset("Abirate/english_quotes", split="train[:32]") # 32 lines
data = data.map(lambda samples: tokenizer(samples['quote']), batched=True)
model._hf_peft_config_loaded = True  # silence a warning from HF trainer

trainer = transformers.Trainer(
    model=model, train_dataset=data,
    args=transformers.TrainingArguments(
        per_device_train_batch_size=2, gradient_accumulation_steps=1,
        # note: if you want larger batch size, increase gradient_accumulation_steps
        warmup_steps=250, max_steps=100, learning_rate=2e-4, fp16=True,
        logging_steps=1, output_dir='outputs', report_to="tensorboard"),
    data_collator=transformers.DataCollatorForLanguageModeling(tokenizer, mlm=False)
)
# if you see cache warnings, set `model.config.use_cache = False` to silence them. Please re-enable for inference!

# kept the optional demonstration disabled before the Python experiment
run_quote_demo = False
if run_quote_demo:
    trainer.train()

# NOTE: this is just an example! you do not have to wait for this progressbar to finish :)

Map:   0%|          | 0/32 [00:00<?, ? examples/s]

### Final task: *actually* train the model (3 points)

Your task is to fine-tune the model to _generate python code_. Please use the above examples for inspiration. More specifically,

* __dataset:__ use [codeparrot-clean](https://huggingface.co/datasets/codeparrot/codeparrot-clean) or any other data containing python code. Since you do not need much data for this excercise, it is enough to use just shorter validation subset of `codeparrots`
* __preprocessing:__ select python code based on file extentions (.py)  (may skip in case of codeparrot - it is 100% python)
* __short lines:__ please take the first 512 characters of each line
* __adapter type:__ please use LoRA as defined above __plus at least one of:__
   - extra adapter on lm_head
   - extra adapter on MLP components (mlp.*)
   - trainable input embeddings (requires tweaking memory usage)

* __training:__ you do not have to train to convergence. If all goes well, your model should `.generate` code after 500 steps. Please use batch size of at least 4 (4 x 1 x 512 tokens) using `gradient_accumulation_steps=4`.


Note: the peft library also has LoRA implementation. However, we ask that for this assignment you show at least one complete training run with your own LoRA code.

__Alternative assignment:__ Instead of doing python code, feel free to substitute the task with any other dataset, e.g. your favorite artist or podcast, as long as it's ethical. If you choose your own task, please show examples of what your model learned - or did not learn, akin to the code examples below.

### numerical checks and completed retry

The diagnostic cell below checked the reloaded model before the retry. It found no non-finite adapter values; the logits and probe loss were finite. Its probe loss was 1.3001. It did not update any weights.

The final LoRA experiment completed all 500 steps, and the comparison table below contains the actual before/after generations. Loss and gradient checks remained enabled during training, and the final logits and loss passed the finiteness checks. The final probe loss was 3.1460. The before-training and final probes used different texts, so these two values were not a before/after quality comparison.

The optional quote-training demonstration remained disabled. Adapter weights were saved every 50 steps and at the end of the retry. These adapter-only files did not include optimizer or scheduler state. The earlier failed run and its diagnosis are summarized in the report below.


In [20]:
# checked current adapters and predictions without updating weights
bad_adapters = [
    name for name, param in model.named_parameters()
    if ('adapter_A' in name or 'adapter_B' in name)
    and not torch.isfinite(param).all().item()
]
print('non-finite adapters:', bad_adapters)
was_training = model.training
model.eval()
probe = tokenizer(
    'import torch\nx = torch.tensor([1, 2, 3])\nprint(x)\n',
    return_tensors='pt', return_token_type_ids=False
).to(device)
with torch.no_grad():
    probe_logits = model(**probe, use_cache=False).logits
    probe_loss = F.cross_entropy(
        probe_logits[:, :-1].float().reshape(-1, probe_logits.shape[-1]),
        probe['input_ids'][:, 1:].reshape(-1)
    )
print('logits dtype:', probe_logits.dtype)
print('finite logits:', torch.isfinite(probe_logits).all().item())
print('probe loss:', probe_loss.item())
print('finite loss:', torch.isfinite(probe_loss).item())
model.train(was_training)
del probe_logits, probe_loss


non-finite adapters: []
logits dtype: torch.float32
finite logits: True
probe loss: 1.3001434803009033
finite loss: True


In [ ]:
prompts =  ['', 'import', 'from', 'while', 'try', 'if', 'for', 'torch']  # feel free to add a few more that are not 100% assiciated with Python

# generate baseline samples with the selected prompts before finetuning
# please feel free to use transformers.Trainer (as above) or your custom training code
# after the training concludes, please show examples of text generated by your model. It is expected to look like Python code fragments
# print the generation examples nicely (suggestion: use pandas or HTML) for easier comparison
# note: your LoRA-enhanced model can run generation the same way as the non-trained model (above)

from itertools import islice
from pathlib import Path

# reset adapters for the independent retry
if 'trainer' in globals():
    del trainer
torch.manual_seed(1337)
for module in model.modules():
    if isinstance(module, LoRALayer):
        nn.init.kaiming_uniform_(module.adapter_A, a=5 ** 0.5)
        nn.init.zeros_(module.adapter_B)
model.zero_grad(set_to_none=True)

def generate_code_samples():
    model.eval()
    model.config.use_cache = True
    samples = []
    with torch.no_grad():
        for prompt in prompts:
            if prompt:
                inputs = tokenizer(prompt, return_tensors='pt', return_token_type_ids=False).to(device)
            else:
                start_id = tokenizer.bos_token_id
                if start_id is None:
                    start_id = tokenizer.eos_token_id
                inputs = {
                    'input_ids': torch.tensor([[start_id]], device=device),
                    'attention_mask': torch.ones((1, 1), dtype=torch.long, device=device)
                }
            generated = model.generate(
                **inputs, max_new_tokens=128, do_sample=False,
                pad_token_id=tokenizer.pad_token_id
            )
            continuation = tokenizer.decode(
                generated[0, inputs['input_ids'].shape[1]:], skip_special_tokens=True
            )
            samples.append(prompt + continuation)
    return samples

before_samples = generate_code_samples()
for prompt, sample in zip(prompts, before_samples):
    print(f'BEFORE | prompt = {prompt!r}\n{sample}\n')

# streamed a small part of the separate CodeParrot validation repository
code_stream = datasets.load_dataset(
    'codeparrot/codeparrot-clean-valid', split='train', streaming=True
).shuffle(seed=1337, buffer_size=1000)
code_texts = [
    item['content'][:512] for item in islice(
        (item for item in code_stream if item['content'].strip()), 2000
    )
]
code_data = datasets.Dataset.from_dict({'text': code_texts})
def tokenize_code(examples):
    return tokenizer(examples['text'], truncation=True, max_length=512, return_token_type_ids=False)
code_data = code_data.map(tokenize_code, batched=True, remove_columns=['text'])
code_data = code_data.filter(lambda example: len(example['input_ids']) > 1)
print('Training snippets:', len(code_data))

# stopped at the first non-finite loss instead of hiding it in logs
class FiniteLossTrainer(transformers.Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        if not inputs['labels'][:, 1:].ne(-100).any().item():
            raise FloatingPointError('This batch has no target tokens.')
        loss, outputs = super().compute_loss(model, inputs, return_outputs=True, **kwargs)
        if not torch.isfinite(loss).all().item():
            raise FloatingPointError('Non-finite training loss; stopped before backward.')
        return (loss, outputs) if return_outputs else loss

# checked gradients and saved small adapter-only recovery files
class AdapterCheckpoints(transformers.TrainerCallback):
    def on_pre_optimizer_step(self, args, state, control, model=None, **kwargs):
        checks = [
            torch.isfinite(param.grad).all()
            for param in model.parameters() if param.requires_grad and param.grad is not None
        ]
        if not checks or not torch.stack(checks).all().item():
            raise FloatingPointError('Missing or non-finite gradients; stopped before the update.')

    def on_step_end(self, args, state, control, model=None, **kwargs):
        if state.global_step % 50 == 0:
            weights = {
                name: param.detach().cpu().clone()
                for name, param in model.named_parameters() if param.requires_grad
            }
            if not all(torch.isfinite(value).all().item() for value in weights.values()):
                raise FloatingPointError('Non-finite adapter weights; checkpoint was not saved.')
            folder = Path('checkpoints_code_retry')
            folder.mkdir(exist_ok=True)
            torch.save(weights, folder / f'adapters_step_{state.global_step}.pt')

model.config.use_cache = False
model.train()
# used four accumulated microbatches as required by the final task
code_trainer = FiniteLossTrainer(
    model=model,
    train_dataset=code_data,
    args=transformers.TrainingArguments(
        per_device_train_batch_size=1, gradient_accumulation_steps=4,
        warmup_steps=250, max_steps=500, learning_rate=2e-4,
        fp16=False, bf16=use_bf16, logging_nan_inf_filter=False,
        logging_steps=10, output_dir='outputs_code_retry', report_to='tensorboard',
        save_strategy='no', seed=1337
    ),
    data_collator=transformers.DataCollatorForLanguageModeling(tokenizer, mlm=False),
    callbacks=[AdapterCheckpoints()]
)
training_result = code_trainer.train()
print('Training loss:', training_result.training_loss)
# checked final logits before accepting the generated comparison
model.eval()
probe = tokenizer('import torch\nprint(torch.zeros(3))\n', return_tensors='pt',
                  return_token_type_ids=False).to(device)
with torch.no_grad():
    probe_output = model(**probe, labels=probe['input_ids'], use_cache=False)
assert torch.isfinite(probe_output.logits).all().item(), 'Non-finite final logits.'
assert torch.isfinite(probe_output.loss).item(), 'Non-finite final probe loss.'
print('Final probe loss:', probe_output.loss.item())
del probe_output
after_samples = generate_code_samples()
# saved only the small custom adapters, not the full quantized model
torch.save(
    {name: param.detach().cpu() for name, param in model.named_parameters() if param.requires_grad},
    'python_lora_retry_adapters.pt'
)

[transformers] Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://hug

BEFORE | prompt = ''
HumanHumanHumanHuman
Human: 以下关于Human的描述中，哪些是正确的？
A. Human类继承自object类
B. Human类继承自object类
C. Human类继承自object类
D. Human类继承自object类
答案: A

Human类继承自object类
Human类继承自object类
Human类继承自object类
Human类继承自object类
Human类继承自object类
Human类继承自object类
Human类继承自object类
Human类继承自object类
Human类继承自object类
Human类继承自object类

BEFORE | prompt = 'import'
import { async, ComponentFixture, TestBed } from '@angular/core/testing';

import { AdminDashboardComponent } from './admin-dashboard.component';
import { RouterTestingModule } from '@angular/router/testing';
import { HttpClientTestingModule } from '@angular/common/http/testing';
import { FormsModule } from '@angular/forms';
import { AdminService } from '../admin.service';
import { AdminServiceMock } from '../admin.service.mock';
import { AdminDashboardService } from './admin-dashboard.service';
import { AdminDashboardServiceMock } from './admin-dashboard.service.mock';
import { AdminDashboardComponentMock } from './admin-dashboard.comp

README.md:   0%|          | 0.00/401 [00:00<?, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/2000 [00:00<?, ? examples/s]

... (2 outputs left out)

Training loss: 1.2517252216339112


[transformers] Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Final probe loss: 3.1460249423980713


[transformers] Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://hug

In [ ]:
# This template helps to compare generated code samples in pretty table form
# feel free to present your work in other forms

from IPython.display import HTML, display
from html import escape
table_template = """<table style="border:1px solid black" >
  <tr>
    <th style="text-align: center; border:1px solid black">PROMPT</th>
    <th style="text-align: center; border:1px solid black">BEFORE</th>
    <th style="text-align: center; border:1px solid black">AFTER</th>
  </tr>
{}
</table>"""

row_template = '''  <tr>
    <td style="width:20%; border:1px solid black"><pre align="left">`{}`</pre></td>
    <td style="width:40%; border:1px solid black"><pre align="left">{}</pre></td>
    <td style="width:40%; border:1px solid black"><pre align="left">{}</pre></td>
  </tr>'''

rows = []

for prompt, before, after in zip(prompts, before_samples, after_samples):
    # escaped generated text before displaying it as HTML
    rows.append(row_template.format(escape(prompt), escape(before), escape(after)))

display(HTML(table_template.format('\n'.join(rows))))

PROMPT,BEFORE,AFTER
``,HumanHumanHumanHuman Human: 以下关于Human的描述中，哪些是正确的？ A. Human类继承自object类 B. Human类继承自object类 C. Human类继承自object类 D. Human类继承自object类 答案: A Human类继承自object类 Human类继承自object类 Human类继承自object类 Human类继承自object类 Human类继承自object类 Human类继承自object类 Human类继承自object类 Human类继承自object类 Human类继承自object类 Human类继承自object类,HumanHumanHumanHuman' is a common phrase used to describe a person's character or personality. It is often used to describe a person's character or personality in a way that is not considered to be a positive or negative or neutral or neutral or neutral or neutral or neutral or neutral or neutral or neutral or neutral or neutral or neutral or neutral or neutral or neutral or neutral or neutral or neutral or neutral or neutral or neutral or neutral or neutral or neutral or neutral or neutral or neutral or neutral or neutral or neutral or neutral or neutral or neutral or neutral or neutral or neutral or neutral or neutral or neutral or neutral or neutral or neutral or neutral
`import`,"import { async, ComponentFixture, TestBed } from '@angular/core/testing'; import { AdminDashboardComponent } from './admin-dashboard.component'; import { RouterTestingModule } from '@angular/router/testing'; import { HttpClientTestingModule } from '@angular/common/http/testing'; import { FormsModule } from '@angular/forms'; import { AdminService } from '../admin.service'; import { AdminServiceMock } from '../admin.service.mock'; import { AdminDashboardService } from './admin-dashboard.service'; import { AdminDashboardServiceMock } from './admin-dashboard.service.mock'; import { AdminDashboardComponentMock } from './admin-dashboard.component.mock'; import { AdminDashboardModule } from './admin",import os import sys import time import json import requests import argparse import logging import logging.handlers import threading import traceback import datetime import re import socket import ssl import urllib import urllib2 import httplib import urlparse import base64 import hashlib import hmac import random import string import Queue import subprocess import signal import platform import psutil import shutil import tempfile import zipfile import tarfile import gzip import bz2 import lzma import zlib import gzip import bz2 import lzma import
`from`,from \eqref{eq:decomp} and \eqref{eq:decomp2} that \begin{equation}\label{eq:decomp3} \begin{aligned} &\int_{\mathbb{R}^d}\left(\frac{\partial^2}{\partial x_1^2}+\cdots+\frac{\partial^2}{\partial x_d^2}\right)\left(\frac{\partial^2}{\partial x_1^2}+\cdots+\frac{\partial^2}{\partial x_d^2}\right)\left(\frac{\,from django.db import models from django.contrib.auth.models import User from django.utils import timezone from django.utils.text import slugify from django.utils.translation import ugettext_lazy as _ from django.core.urlresolvers import reverse from django.core.exceptions import ValidationError from django.conf import settings from django.db.models.signals import post_save from django.dispatch import receiver from rest_framework.authtoken.models import Token from django.contrib.auth.models import AbstractUser from django.contrib.auth import get_user_model from django.db.models import Q from django.db.models.signals import post_save from django.dispatch import receiver from django.utils import
`while`,while \n\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t,"while (1) { if (Serial.available() > 0) { char c = Serial.read(); if (c == '\n') { // End of line, process the command Serial.print(""Received: ""); Serial.println(command); // Process the command processCommand(command); // Reset the command buffer command = """"; } else { // Append the character to the command buffer command += c; } } } delay(10); // Small delay to prevent busy-waiting } ``` ### 3. **Using a State Machine"
`t

### report

I trained 16 virtual prompt tokens on the fox sentence using my own embedding layer and HuggingFace PEFT. In the saved run, the final losses were 0.0466 and 0.0762, respectively, both below 0.1. Both models generated "A quick brown fox did not jump over the lazy dog", unlike the original model. The embedding checks, LoRA unit tests and adapter gradient checks passed.

For Python generation, I used my own rank-8 LoRA on Q/K/V and the MLP down projections, with 10,027,008 trainable parameters. I selected 2,000 Python snippets from CodeParrot clean validation and kept the first 512 characters of each. I used 500 optimizer steps, microbatch size 1, gradient accumulation 4, learning rate 2e-4 and 250 warmup steps. The 500 steps followed the assignment's recommendation, not a strict minimum.

The first LoRA run used FP16 and failed numerically: logged losses became zero from step 290 onward, and all eight final continuations consisted of repeated exclamation marks. Diagnostics in that session found non-finite adapter values in all 36 decoder layers, non-finite logits and a NaN probe loss. This confirmed a numerical failure, but did not identify the first failing operation or prove that FP16 was the only cause.

For the retry, I disabled FP16 and used the BF16 compute setting selected by the environment on a Tesla T4. I kept the data, learning rate, step count and effective batch size unchanged. I added checks for non-finite loss and gradients, disabled NaN/Inf filtering in the logs and saved adapter-only checkpoints every 50 steps. The retry completed 500 steps in 2 hours 5 minutes 11 seconds. The average training loss was 1.2517, and the last logged loss was 1.2233. The numerical checks passed, including the final logits and probe loss checks; the earlier numerical failure did not recur in this run.

The same eight prompts were used before and after training, with greedy decoding and a limit of 128 new tokens. After training, the continuations for `import`, `from`, `try`, `if`, `for` and `torch` looked like Python code instead of the formulas, other languages or repetitions seen before training. The result was not consistent across all prompts: `while` produced C++/Arduino code, and the empty prompt produced repetitive prose. Some Python outputs also contained repeated imports, older APIs or unfinished lines.

Overall, the retry showed a qualitative shift toward Python generation, not verified program correctness. I did not execute the generated code or measure held-out code quality, and the training loss was not a test metric. The two diagnostic probe losses used different texts and were not directly comparable. The optional bonus was not attempted.


If you reach this: congratulations! you've completed everything in this practice session.

If you want to dig deeper, try to implement prompt-tuning (for bonus points!).
You can read more about prompt tuning variants in paper [1](https://arxiv.org/abs/2104.08691) or paper [2](https://arxiv.org/abs/2101.00190). Both versions can be implemented by passing trainable prompts as `model.forward(..., past_key_values=your_prompts)`.



### Read more

* How post-training quantization works: https://arxiv.org/abs/2208.07339
* An overview of running large models: https://huggingface.co/docs/accelerate/package_reference/big_modeling
* A general library for different adapter types: https://adapterhub.ml/


### [extra info] Running other models.

This notebook's code can run with other models of similar size, such as [Falcon-7B](https://huggingface.co/tiiuae/falcon-7b), [OPT-6.7B](https://huggingface.co/facebook/opt-6.7b) or [BLOOM-7.1B](https://huggingface.co/bigscience/bloom-7b1). However, they will require minor code tweaks:
1. change the model name in `AutoModelForCausalLM.from_pretrained()` __and__ `AutoTokenizer`
2. In the prompt tuning code, change `model.model.embed_tokens` to refer to the target model's word embeddings. Simply `print(model)` to navigate to them.
3. Change code to add Lora layers - specifically where you what the transformer block components, since those components now have different names.